In [ ]:
from __future__ import annotations

import torch

import torch.nn as nn
import torch.nn.functional as F

import torchvision
from torchvision import datasets, transforms

from pathlib import Path
import matplotlib.pyplot as plt

# from torch_tensors_images.py assignment
def get_dataset(root: Path) -> datasets.CIFAR10:
    """
    Returns the CIFAR-10 training set.

    The first time you run this, it will download the dataset to `root`.
    """
    root.mkdir(parents=True, exist_ok=True)
    return datasets.CIFAR10(root=str(root), train=True, download=True, transform=torchvision.transforms.ToTensor())


# define the network
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # conv layer: 
        #   in = 3 since CIFAR-10 images have 3 color channels (RGB)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5)
        # fully connected layers
        self.fc1 = nn.Linear(64 * 2 * 2, 200)
        self.fc2 = nn.Linear(200, 10)

    def forward(self, x):  

        # used the photo's architecture as reference

        # conv1 to relu 
        x = F.relu(self.conv1(x))
        # maxpool(kernel=3, stride=3)
        x = F.max_pool2d(x, kernel_size=3, stride=3)

        # conv2 to relu 
        x = F.relu(self.conv2(x))
        # maxpool(kernel=2, stride=2)
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        x = x.view(-1, 256) 

        # fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x



    # data_root = Path(__file__).resolve().parent / "data"

    # copied the dataset into the root of the project
data_root = Path("./data") # alt way since it kept saying __file__ is not defined
dataset = get_dataset(data_root)
data_loader = torch.utils.data.DataLoader(dataset,batch_size=64,shuffle=True)
images, labels = next(iter(data_loader))
# Create network
net = Net()
# Forward pass
output = net(images)
print(f"Input tensor shape: {images.shape}")    # [64, 3, 32, 32]
print(f"Output tensor shape: {output.shape}")   # [64, 10]
optimizer = torch.optim.Adam(net.parameters(), lr = 0.001)
criterion = nn.CrossEntropyLoss() # another name for loss function
# training loop
for epoch in range(5): # outer for loop layer for reshuffling
    for images, labels  in data_loader: # inner loop for training
        optimizer.zero_grad()
        output = net(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}, loss: {loss.item():.4f}")    

Files already downloaded and verified
Input tensor shape: torch.Size([64, 3, 32, 32])
Output tensor shape: torch.Size([64, 10])
Epoch 1, loss: 1.5624
Epoch 2, loss: 1.7511
Epoch 3, loss: 0.9237
Epoch 4, loss: 1.2132
Epoch 5, loss: 1.0808


In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=2)
        self.conv3 = nn.Conv2d(64, 256, kernel_size=5, stride=2)
        self.conv4 = nn.Conv2d(256, 256, kernel_size=5, stride=2)

    def forward(self, x):  
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        return x
    
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(256, 128, kernel_size=2)
        self.conv2 = nn.Conv2d(128, 32, kernel_size=5)
        self.conv3 = nn.Conv2d(32, 16, kernel_size=5, padding=1)
        self.conv4 = nn.Conv2d(16, 3, kernel_size=6, padding=3)
                               
    def forward(self, x):  
        x = F.interpolate(x, scale_factor=4)
        x = F.relu(self.conv1(x))
        x = F.interpolate(x, scale_factor=3)
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=3)
        x = F.relu(self.conv3(x))
        x = F.interpolate(x, scale_factor=3)
        x = F.relu(self.conv4(x))
        return x[:, :, :32, :32] # crop to 32x32

enc = Encoder()
dec = Decoder()

z = enc(images)
print(z.shape, "Encoder output shape")

recon = dec(z)
print(recon.shape, "Reconstruction shape")

# important, reduce your dataset to only cat imaged (use test_dataset.classes to find)
# the index of cat class, then filter the dataset to only include those images) and train the autoencoder on that subset of data

# Make sure the output is the same as the input
# Write the nested for loops over epoch and batches of data
# Prepare a batch of noisy and claen images
# inspect the noisy image, the std of noise should be small enough so that the image is still recognizable.
# Use MSE loss and Adam optimizer to train the autoencoder
# Monitor the training loss (you may need to reduce the learning rate and increase the number of epochs to see the loss decrease)




torch.Size([16, 256, 1, 1]) Encoder output shape
torch.Size([16, 3, 32, 32]) Reconstruction shape


In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr = 0.001)
criterion = nn.MSELoss() # another name for loss function

# training loop
for epoch in range(5): # outer for loop layer for reshuffling
    for images, labels  in data_loader: # inner loop for training
        optimizer.zero_grad()
        output = enc(images)
        loss = criterion(output, images)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}, loss: {loss.item():.4f}")